# Standard vs. Agentic contract-obligation benchmark

Use the official public CU CLI for service operations and local Python for preparation/evaluation. The checked-in REPORT.md and qualitative review are historical evidence, not outputs of these edited cells. Missing usage/timing is unknown, never zero. Paid execution is off by default.

In [ ]:
import json, os, shutil, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from IPython.display import Markdown, display

PROJECT = Path.cwd().resolve()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
elif PROJECT.name != '06-Contract-Obligation-Golden-Set':
    PROJECT = PROJECT / 'examples' / '06-Contract-Obligation-Golden-Set'
if not (PROJECT / 'dataset' / 'selection_manifest.json').is_file():
    raise FileNotFoundError('Start from the repository, example, or notebook directory')
REPO = PROJECT.parents[1]
local_cli = REPO / '.venv' / ('Scripts' if os.name == 'nt' else 'bin') / ('cu.exe' if os.name == 'nt' else 'cu')
CU = os.environ.get('CU_GOLDEN_CLI') or (str(local_cli) if local_cli.is_file() else shutil.which('cu'))
if not CU:
    raise FileNotFoundError('Install the official CU CLI; see the repository README')
version = subprocess.run([CU, '--version'], cwd=REPO, stdin=subprocess.DEVNULL, check=True, capture_output=True, text=True)
if 'version 0.1.0b1' not in version.stdout:
    raise RuntimeError('This notebook targets official cu 0.1.0b1; set CU_GOLDEN_CLI to that executable')
RUN_LIVE = os.environ.get('CU_GOLDEN_RUN_LIVE') == '1'
RUN_ID = datetime.now(timezone.utc).strftime('cli_%Y%m%dT%H%M%SZ_') + uuid4().hex[:8]
NEW_RESULTS = PROJECT / 'test_results' / RUN_ID
METRICS = PROJECT / 'evaluation' / 'output' / RUN_ID
SAMPLES = PROJECT / 'samples' / 'cli_inputs'
STANDARD_ID, AGENTIC_ID = 'golden_standard_' + RUN_ID, 'golden_agentic_' + RUN_ID
STANDARD_RESULTS = NEW_RESULTS / 'standard' if RUN_LIVE else PROJECT / 'test_results' / 'standard'
AGENTIC_RESULTS = NEW_RESULTS / 'agentic' if RUN_LIVE else PROJECT / 'test_results' / 'agentic'
print(f'Project: {PROJECT}\nCLI: {CU}\nPaid execution approved: {RUN_LIVE}\nNew metrics: {METRICS}')

## 1. Offline preparation and validation

Download the public checksum-pinned CUAD fixture through example 05 first if needed. No private framework is required. The CLI owns authentication through its profiles / CU_* configuration; this notebook does not load .env or change resource/model defaults.

In [ ]:
subprocess.run([sys.executable, str(PROJECT/'scripts'/'prepare_dataset.py'), '--output', str(SAMPLES), '--manifest-output', str(SAMPLES.with_suffix('.manifest.json'))], cwd=REPO, stdin=subprocess.DEVNULL, check=True)
subprocess.run([sys.executable, str(PROJECT/'scripts'/'build_schemas.py')], cwd=REPO, stdin=subprocess.DEVNULL, check=True)
gold = [json.loads(line) for line in (PROJECT/'ground_truth'/'golden_obligations.jsonl').read_text(encoding='utf-8').splitlines() if line]
print(f"Golden set: {len(gold)} contracts, {sum(len(row['obligations']) for row in gold)} obligations")

In [ ]:
standard_schema = PROJECT/'schemas'/'contract_obligations_standard_v1.json'
agentic_schema = PROJECT/'schemas'/'contract_obligations_agentic_v1.json'
subprocess.run([CU, 'analyzer', 'validate', str(standard_schema), '--api-version', '2025-11-01'], cwd=REPO, stdin=subprocess.DEVNULL, check=True)
subprocess.run([CU, 'analyzer', 'validate', str(agentic_schema), '--api-version', '2026-06-01-preview'], cwd=REPO, stdin=subprocess.DEVNULL, check=True)

## 2. Local plans; paid execution only after explicit approval

The two dry-runs make no service calls or writes. The source directory contains only the ten contracts; its provenance manifest is outside it, so no --pattern filter is needed. Setting CU_GOLDEN_RUN_LIVE=1 before starting Jupyter approves all 20 paid analyses and enables --yes. Each mode gets a new analyzer ID and output directory. Schemas and stderr logs are preserved. A nonzero exit stops execution; inspect the report and evaluate partial runs separately if desired. New analyzers are retained: delete only your newly created IDs when finished.

In [ ]:
standard_command = [CU, 'analyze', '--source', str(SAMPLES), '--recursive', '--analyzer', STANDARD_ID, '--json', '--output-dir', str(NEW_RESULTS/'standard'), '--report-file', str(NEW_RESULTS/'standard'/'report.json'), '--api-version', '2025-11-01', '--concurrency', '5', '--on-existing', 'error', '--usage', '--time']
agentic_command = [CU, 'analyze', '--source', str(SAMPLES), '--recursive', '--analyzer', AGENTIC_ID, '--json', '--output-dir', str(NEW_RESULTS/'agentic'), '--report-file', str(NEW_RESULTS/'agentic'/'report.json'), '--api-version', '2026-06-01-preview', '--concurrency', '5', '--on-existing', 'error', '--usage', '--time']
subprocess.run(standard_command + ['--dry-run'], cwd=REPO, stdin=subprocess.DEVNULL, check=True)
subprocess.run(agentic_command + ['--dry-run'], cwd=REPO, stdin=subprocess.DEVNULL, check=True)

In [ ]:
if RUN_LIVE:
    NEW_RESULTS.mkdir(parents=True, exist_ok=False)
    subprocess.run([CU, 'analyzer', 'create', '--name', STANDARD_ID, '--schema', str(standard_schema), '--api-version', '2025-11-01'], cwd=REPO, stdin=subprocess.DEVNULL, check=True)
    with (NEW_RESULTS/'standard.schema.json').open('x', encoding='utf-8') as schema_file:
        subprocess.run([CU, 'analyzer', 'show', STANDARD_ID, '--api-version', '2025-11-01'], cwd=REPO, stdin=subprocess.DEVNULL, stdout=schema_file, check=True)
    with (NEW_RESULTS/'standard.stderr.log').open('x', encoding='utf-8') as log:
        subprocess.run(standard_command + ['--yes'], cwd=REPO, stdin=subprocess.DEVNULL, stderr=log, check=True)
    subprocess.run([CU, 'analyzer', 'create', '--name', AGENTIC_ID, '--schema', str(agentic_schema), '--api-version', '2026-06-01-preview'], cwd=REPO, stdin=subprocess.DEVNULL, check=True)
    with (NEW_RESULTS/'agentic.schema.json').open('x', encoding='utf-8') as schema_file:
        subprocess.run([CU, 'analyzer', 'show', AGENTIC_ID, '--api-version', '2026-06-01-preview'], cwd=REPO, stdin=subprocess.DEVNULL, stdout=schema_file, check=True)
    with (NEW_RESULTS/'agentic.stderr.log').open('x', encoding='utf-8') as log:
        subprocess.run(agentic_command + ['--yes'], cwd=REPO, stdin=subprocess.DEVNULL, stderr=log, check=True)
else:
    print('Skipped paid calls. CU_GOLDEN_RUN_LIVE=1 explicitly approves a new run.')

## 3. Fail-closed offline evaluation

Native root contents and legacy envelopes are supported. Full source-relative paths and extensions identify results; unrelated/duplicate/malformed results are errors. Every expected gold document remains in the denominator, including missing/failed/skipped inputs. CLI report status is read from report.json when present. Numeric usage/latency absent from structured results remains null, not zero. New reports never inherit the historical qualitative verdict.

In [ ]:
evaluator = PROJECT/'evaluation'/'evaluate.py'
if STANDARD_RESULTS.is_dir() and AGENTIC_RESULTS.is_dir():
    subprocess.run([sys.executable, str(evaluator), '--standard-results', str(STANDARD_RESULTS), '--agentic-results', str(AGENTIC_RESULTS), '--samples', str(SAMPLES), '--output', str(METRICS)], cwd=REPO, stdin=subprocess.DEVNULL, check=True)
else:
    print('Both result directories are required; historical evidence is in the checked-in REPORT.md.')

In [ ]:
if (METRICS/'standard_metrics.json').is_file() and (METRICS/'agentic_metrics.json').is_file():
    standard = json.loads((METRICS/'standard_metrics.json').read_text(encoding='utf-8'))
    agentic = json.loads((METRICS/'agentic_metrics.json').read_text(encoding='utf-8'))
    rows = [('Completion', standard['completion_rate'], agentic['completion_rate']), ('Precision', standard['precision'], agentic['precision']), ('Recall', standard['recall'], agentic['recall']), ('F1', standard['f1'], agentic['f1']), ('Groundedness', standard['quote_groundedness'], agentic['quote_groundedness'])]
    table = ['| Metric | Standard | Agentic | Delta |', '|---|---:|---:|---:|'] + [f'| {name} | {left:.1%} | {right:.1%} | {right-left:+.1%} |' for name, left, right in rows]
    display(Markdown('\n'.join(table)))
    display(Markdown((METRICS/'REPORT.md').read_text(encoding='utf-8')))